# Week 3: Evaluation using reference ground-truth, mitigation, and calibration

You now receive:

- `week3_validation_reference.csv`
- `week3_test_observed.csv`
- `week3_test_reference.csv`

These files let you compare evaluation against the labels available to the modelling pipeline, $Y^{obs}$, and the best available reference labels, $Y^{ref}$.

You now receive the "clean" reference labels (`income_reference`) for the validation and test sets.

Your objective this week is to uncover the **root cause** of the biases and anomalies you hypothesized in Week 2. By comparing your model's performance against the observed labels ($Y^{obs}$) versus the clean reference labels ($Y^{ref}$), you will finally validate (or invalidate!) your previous hypotheses.

Once you have identified the true source of the bias, you will design and implement a mitigation strategy of your choice to correct it.

**Rules**
- Do not train a predictive model on `income_reference`.                                    
- Reference validation may be used for final model selection or calibration only if you explicitly state and defend that design choice.                         
- Reference test is used exactly *once* for your final evaluation.                                         
- Preserve `row_id` when joining files and verify one-to-one joins.

- Now we have 3 new files, validation reference, test observed, test reference
- I assume this means we used the model trained on the training data from last week ?
- I guess we evaluate model on observe and the reference and observe the results

- We are essentially asking does our current trained model on the data from week 2 look fair or unfair depending on what labels we audit against ( ref vs. observed), so not training a new model but evaluating the one we already have

Regel 1: "Do not train a predictive model on income_reference"

Hvorfor: Vi diskuterede dette tidligere i samtalen, men lad mig genopfriske det med den nye kontekst. income_reference findes kun for validation og test (6.000+6.000 personer) — ikke for jeres 18.000 trænings-rows. Det er et bevidst designvalg, der efterligner en meget almindelig, reel situation:

Et firma har en billig, automatisk label for alle deres data (jeres income_observed), men en dyr, omhyggeligt verificeret label kun for en lille stikprøve (jeres income_reference).

Hvis I fik lov at træne på reference-labels, ville I aldrig opleve selve problemet, kurset vil undersøge: hvordan finder og retter man bias, når man kun har adgang til upålidelige labels i den mængde, man skal træne på? At træne direkte på reference ville være at snyde uden om denne centrale udfordring — og desuden er der (i den fiktive virkelighed, øvelsen simulerer) ikke nok reference-data (kun 6.000 rows) til at træne en god model alene.

Regel 4: "Preserve row_id... verify one-to-one joins"

Hvorfor: Ren datahygiejne — sikrer, at når I sammenligner income_observed og income_reference for "samme person", er det faktisk samme person i begge filer, og at ingen personer ved et uheld duplikeres eller mistes under join'et (hvilket ville forvride jeres disagreement-rate-beregninger). I har allerede gjort dette korrekt (validate="1:1" i jeres kode, og bekræftet 6000=6000=6000).

## 1. Loading Data

In [158]:
%pip -q install fairlearn cleanlab scikit-learn pandas matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [159]:
!git clone https://github.com/ADE-17/RAI_Project1_Fairness_Project.git

fatal: destination path 'RAI_Project1_Fairness_Project' already exists and is not an empty directory.


In [160]:
DATA_DIR = "/content/RAI_Project1_Fairness_Project/data"

In [161]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
DATA_DIR = Path("/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/9. sem/Ansvarlig AI/Fairness/Responsible_AI/data2")

train = pd.read_csv(DATA_DIR / "week2_train_observed.csv")
val_obs = pd.read_csv(DATA_DIR / "week2_validation_observed.csv")
val_ref = pd.read_csv(DATA_DIR / "week3_validation_reference.csv")
test_obs = pd.read_csv(DATA_DIR / "week3_test_observed.csv")
test_ref = pd.read_csv(DATA_DIR / "week3_test_reference.csv")

# TODO: assert unique row_id values and exact alignment of feature columns.
print(train.shape, val_obs.shape, val_ref.shape, test_obs.shape, test_ref.shape)

(18000, 15) (6000, 15) (6000, 16) (6000, 15) (6000, 16)


- train has 18 k rows, 14 features
- validation and test has 6 K rows,

## 2. Comparing Yobs and Yref on validation

To start with - Compare $Y^{obs}$ and $Y^{ref}$ on validation and test data. Report overall disagreement, disagreement conditional on each label, group-specific disagreement, and intersections. Inspect which feature regions contain disagreements without assuming the reference label is infallible.

- We only need to compare the validation obs and ref and not test since this is used just once

In [162]:
# TODO: join observed and reference files by row_id.
# TODO: quantify and visualize label disagreement.
# TODO: test the Week 2 hypotheses without reading the new metadata.

test only row id overlap - not additional features
Hvad koden faktisk verificerer:

assert val_obs["row_id"].is_unique — tjekker, at der ikke findes duplikerede row_id'er inde i val_obs alene (fx at person 4231 ikke optræder to gange i samme fil)
assert val_ref["row_id"].is_unique — samme tjek, men for val_ref
validate="1:1" i selve .merge() — dette er faktisk det stærkeste tjek: det garanterer, at hver row_id i val_obs matcher præcis én row_id i val_ref, og omvendt. Hvis der var duplikater eller hvis nogle row_id'er kun fandtes i den ene fil, ville merge() fejle med det samme (kaste en MergeError)
len(val_merged) != len(val_obs) — et ekstra, menneskelæsbart sikkerhedstjek: bekræfter, at alle 6000 rows fra val_obs faktisk fandt en makker i val_ref (ingen blev tabt undervejs)

In [163]:
# TODO: join observed and reference files by row_id.
val_merged = val_obs.merge(
    val_ref[["row_id", "income_reference"]],
    on="row_id",
    how="inner",
    validate="1:1",  # raises immediately if the join isn't one-to-one
)

assert val_obs["row_id"].is_unique, "val_obs has duplicate row_ids"
assert val_ref["row_id"].is_unique, "val_ref has duplicate row_ids"

print(f"val_obs rows: {len(val_obs)}, val_ref rows: {len(val_ref)}, joined rows: {len(val_merged)}")
if len(val_merged) != len(val_obs):
    print("WARNING: join dropped rows -- val_obs and val_ref do not cover the same row_ids")

val_obs rows: 6000, val_ref rows: 6000, joined rows: 6000


check if all other features are mathcing (not income)

In [164]:
# 1. Er det samme row_id-mængde?
print("Same row_ids:", set(test_ref["row_id"]) == set(test_obs["row_id"]))

# 2. Join dem og tjek at ALLE feature-værdier er identiske for hver person
feature_cols = ["AGEP", "COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "WKHP", 
                "SEX", "RAC1P", "SEX_group", "RACE_group", "SURVEY_BATCH", "income_observed"]

merged_check = test_obs.merge(test_ref, on="row_id", suffixes=("_obs", "_ref"))

mismatches = 0
for col in feature_cols:
    col_obs = f"{col}_obs" if f"{col}_obs" in merged_check.columns else col
    col_ref = f"{col}_ref" if f"{col}_ref" in merged_check.columns else col
    if col_obs in merged_check.columns and col_ref in merged_check.columns:
        diff = (merged_check[col_obs] != merged_check[col_ref]).sum()
        if diff > 0:
            print(f"MISMATCH in {col}: {diff} rows differ")
            mismatches += diff

if mismatches == 0:
    print("All features match exactly across test_obs and test_ref for shared columns.")

Same row_ids: True
All features match exactly across test_obs and test_ref for shared columns.


- All rows merged fine without duplicates or errors

In [165]:
# TODO: quantify and visualize label disagreement.
val_merged["disagree"] = val_merged["income_observed"] != val_merged["income_reference"]

overall_disagreement = val_merged["disagree"].mean()
print(f"Overall disagreement rate: {overall_disagreement:.4f} "
      f"({val_merged['disagree'].sum()} of {len(val_merged)} rows)")

Overall disagreement rate: 0.0403 (242 of 6000 rows)


- 4 % percent of the rows disagree, not a lot

In [166]:
print("Disagreement rate, conditional on income_observed:")
print(val_merged.groupby("income_observed")["disagree"].agg(
    disagreement_rate="mean", n="count"
))

Disagreement rate, conditional on income_observed:
                 disagreement_rate     n
income_observed                         
0                         0.053080  3994
1                         0.014955  2006


- The n column contans the total number of observations for either 0 or 1
- The disagreement is then how many percentages within that group disagree
- So within the low income folk 5 % of the observations disagree, so 5 % of the 3994


In [167]:
print("Disagreement rate by SEX_group:")
print(val_merged.groupby("SEX_group")["disagree"].agg(disagreement_rate="mean", n="count"))

print("\nDisagreement rate by RACE_group:")
print(val_merged.groupby("RACE_group")["disagree"].agg(disagreement_rate="mean", n="count").sort_values("disagreement_rate", ascending=False))

print("\nDisagreement rate by SURVEY_BATCH:")
print(val_merged.groupby("SURVEY_BATCH")["disagree"].agg(disagreement_rate="mean", n="count"))

Disagreement rate by SEX_group:
           disagreement_rate     n
SEX_group                         
Female              0.072546  2812
Male                0.011920  3188

Disagreement rate by RACE_group:
                      disagreement_rate     n
RACE_group                                   
Asian                          0.094340   318
American Indian                0.050000    20
Two or more races              0.044776   134
White                          0.040400  4703
Black                          0.022814   526
Other race                     0.010676   281
AIAN specified/other           0.000000    15
Pacific Islander               0.000000     3

Disagreement rate by SURVEY_BATCH:
              disagreement_rate    n
SURVEY_BATCH                        
B01                    0.024961  641
B02                    0.032258  620
B03                    0.026480  642
B04                    0.049927  681
B05                    0.042641  727
B06                    0.041537  963
B0

- 7 % (of the observations disagree between obs and ref for women compared to 1 % for men
- Higher disagrement rate for asians compared to black and white. almost twice the disagreement for white compared to black
- A higher disagreement rate for later batches after B03 which coincides with observations we did in week 2
- The disagrement rate for race does produce a consistent pattern. American indian and asian are both higher than black and white
- With regards to batches we do say a doubling from B03 to B07.
- The largest disrepancy is between men and women with women having 6x higher disagreement rate¨
-

In [168]:
val_merged["under_labeled"] = (val_merged["income_observed"] == 0) & (val_merged["income_reference"] == 1)
val_merged["over_labeled"] = (val_merged["income_observed"] == 1) & (val_merged["income_reference"] == 0)

print("Directional error rate by SEX_group (both as % of that group's total rows):")
print(val_merged.groupby("SEX_group")[["under_labeled", "over_labeled"]].agg(["mean", "sum"]))

Directional error rate by SEX_group (both as % of that group's total rows):
          under_labeled      over_labeled    
                   mean  sum         mean sum
SEX_group                                    
Female         0.066501  187     0.006046  17
Male           0.007842   25     0.004078  13


- Under labeled is where the label is false in obs but true in ref and vice versa overlabeled is true in obs but false in ref

- We see women are more than 10 x under labeled compared to men, meaning women are underestimated in earnings compared to men in corrupted data

In [169]:
print("Under-labeling rate, by SURVEY_BATCH x SEX_group:")
print(val_merged.groupby(["SURVEY_BATCH", "SEX_group"])["under_labeled"].agg(
    under_labeled_rate="mean", n="count"
).unstack("SEX_group"))

Under-labeling rate, by SURVEY_BATCH x SEX_group:
             under_labeled_rate                n     
SEX_group                Female      Male Female Male
SURVEY_BATCH                                         
B01                    0.029412  0.008130    272  369
B02                    0.052000  0.005405    250  370
B03                    0.044355  0.002538    248  394
B04                    0.059937  0.024725    317  364
B05                    0.068783  0.005731    378  349
B06                    0.077079  0.002128    493  470
B07                    0.090498  0.007407    442  405
B08                    0.077670  0.008565    412  467


## 3. Re-evaluate the model

Re-evaluate the original model twice

For the same scores and decisions, produce two complete audits:

1. against `income_observed`;
2. against `income_reference`.

Any difference is caused by the evaluation target, not by a changed model. Discuss which conclusions reverse or materially change.

In [170]:
# TODO: reconstruct your Week 2 baseline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# not sensitive features
FEATURE_COLUMNS = ["AGEP", "COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "WKHP"]

pre = ColumnTransformer([("num", StandardScaler(), FEATURE_COLUMNS)], remainder="drop")
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
model.fit(train[FEATURE_COLUMNS], train["income_observed"])

# Score and decide on the validation set ONCE -- these same scores/predictions get
# audited twice below, against two different labels. The model and its outputs never change.
score = model.predict_proba(val_merged[FEATURE_COLUMNS])[:, 1]
pred = (score >= 0.5).astype(int)

print(f"Scored {len(pred)} validation rows, {pred.mean():.1%} predicted positive.")

Scored 6000 validation rows, 27.0% predicted positive.


- Model is used on validation merged which contains both obs and ref labels
- The 27 % is just how many rows the model predicted positive but not how many of those predictions were correct against either obs or ref


In [171]:
import numpy as np
from fairlearn.metrics import MetricFrame, selection_rate, count
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                              log_loss, precision_score, recall_score, brier_score_loss)

def false_positive_rate_metric(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    return fp / (fp + tn) if (fp + tn) > 0 else np.nan

def negative_predictive_value(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tn / (tn + fn) if (tn + fn) > 0 else np.nan

metrics = {
    "count": count,
    "selection_rate": selection_rate,   # independence
    "TPR": recall_score,                # separation
    "FPR": false_positive_rate_metric,  # separation
    "PPV": precision_score,             # sufficiency
    "NPV": negative_predictive_value,   # sufficiency
}

In [172]:
def run_audit(y_eval, scores, predictions, sensitive_features, label_name):
    """Full fairness+performance audit of a fixed set of scores/predictions,
    against a given evaluation label. Explicitly names which label was used,
    per the notebook's instruction."""

    print(f"===================== Evaluated against {label_name} =====================")

    # --- Utility (overall performance) ---
    print("\n-- Utility --")
    print(f"Accuracy:          {accuracy_score(y_eval, predictions):.4f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_eval, predictions):.4f}")
    print(f"AUROC:             {roc_auc_score(y_eval, scores):.4f}")
    print(f"Log loss:          {log_loss(y_eval, scores):.4f}")

    # --- Independence / Separation / Sufficiency, by group ---
    mf = MetricFrame(metrics=metrics, y_true=y_eval, y_pred=predictions,
                      sensitive_features=sensitive_features)
    ratio = mf.ratio(method="between_groups")
    print("\n-- By group --")
    print(mf.by_group.round(4))
    print("\n-- Fairness ratios (min/max across groups) --")
    print(f"Independence -- selection_rate ratio: {ratio['selection_rate']:.4f}")
    print(f"Separation   -- TPR ratio: {ratio['TPR']:.4f}   FPR ratio: {ratio['FPR']:.4f}")
    print(f"Sufficiency  -- PPV ratio: {ratio['PPV']:.4f}   NPV ratio: {ratio['NPV']:.4f}")

    # --- Calibration ---
    print(f"\n-- Calibration --")
    print(f"Brier score: {brier_score_loss(y_eval, scores):.4f}")

    print()
    return ratio

In [173]:
A_val = val_merged["SEX_group"]

ratio_obs = run_audit(val_merged["income_observed"], score, pred, A_val, "Y^observed")
ratio_ref = run_audit(val_merged["income_reference"], score, pred, A_val, "Y^reference")

===================== Evaluated against Y^observed =====================

-- Utility --
Accuracy:          0.7615
Balanced accuracy: 0.7080
AUROC:             0.8179
Log loss:          0.4868

-- By group --
            count  selection_rate     TPR     FPR     PPV     NPV
SEX_group                                                        
Female     2812.0          0.2315  0.5080  0.1527  0.4869  0.8579
Male       3188.0          0.3030  0.5637  0.1035  0.8064  0.7286

-- Fairness ratios (min/max across groups) --
Independence -- selection_rate ratio: 0.7640
Separation   -- TPR ratio: 0.9012   FPR ratio: 0.6783
Sufficiency  -- PPV ratio: 0.6038   NPV ratio: 0.8493

-- Calibration --
Brier score: 0.1604

===================== Evaluated against Y^reference =====================

-- Utility --
Accuracy:          0.7702
Balanced accuracy: 0.7242
AUROC:             0.8423
Log loss:          0.4775

-- By group --
            count  selection_rate     TPR     FPR     PPV     NPV
SEX_group    

From the above we observe:
- Independence doesnt change, since it doesnt include the label in its veridct by its formula. Its violated in both instances
- Equalized odds ratio actually flips from FPR going from 0.67 (violation) to 0.89 (no violation) with true labels
- Same with sufficiency, PPV is violated in Yobs and not vilated in Yref.


In [174]:
# TODO: reconstruct your Week 2 baseline.
# TODO: create a reusable audit function accepting y_eval, scores, predictions, and sensitive features.
# TODO: report utility, independence, separation, sufficiency, and calibration twice. Brier is the calibration metric

## 4. Mitigation candidates

HintL Design mitigation candidates

Compare at least four methods you could think of. You may add models. Every intervention requires a  rationale and a documented cost/tradeoff.

In [175]:
# TODO: define the candidates and a common experiment protocol.
# Keep preprocessing, split, random seed, and evaluation consistent.

In [176]:
X_train, y_train = train[FEATURE_COLUMNS], train["income_observed"]
A_train = train["SEX_group"]
B_train = train["SURVEY_BATCH"]

X_val = val_merged[FEATURE_COLUMNS]

def summarize_candidate(name, y_pred, results_list):
    """Evaluate one candidate's predictions against BOTH labels, record ratios for later comparison."""
    row = {"candidate": name,
           "accuracy_obs": accuracy_score(val_merged["income_observed"], y_pred),
           "accuracy_ref": accuracy_score(val_merged["income_reference"], y_pred)}
    for label_name, y_eval in [("obs", val_merged["income_observed"]), ("ref", val_merged["income_reference"])]:
        mf = MetricFrame(metrics=metrics, y_true=y_eval, y_pred=y_pred, sensitive_features=A_val)
        ratio = mf.ratio(method="between_groups")
        for k in ["selection_rate", "TPR", "FPR", "PPV", "NPV"]:
            row[f"{k}_ratio_{label_name}"] = ratio[k]
    results_list.append(row)

results = []
summarize_candidate("1. Baseline (unmitigated)", pred, results)

Hint: `ThresholdOptimizer` changes decisions, not probability calibration. `ExponentiatedGradient` trains a randomized classifier under a constraint. Equalized odds is evaluated relative to the label supplied during fitting.

1. "ThresholdOptimizer changes decisions, not probability calibration"

ThresholdOptimizer ændrer ikke modellens underliggende score/sandsynlighed (fx om en person scores 0.62 eller 0.71) — den ændrer kun, ved hvilken threshold** en person går fra ŷ=0 til ŷ=1, og den kan bruge forskellige thresholds for forskellige grupper. Så hvis modellen giver en kvinde en score på 0.45 og en mand en score på 0.45, kunne ThresholdOptimizer beslutte "kvinder skal bruge threshold 0.40, mænd skal bruge threshold 0.55" — så kvinden bliver klassificeret som positiv, manden som negativ, selvom deres underliggende score er identisk.

Konsekvens: hvis I efter at have brugt ThresholdOptimizer beregner Brier score eller laver et reliability-plot på de rå scores, vil I se præcis samme kalibrering som før — fordi selve scoren aldrig blev rørt. ThresholdOptimizer løser ikke et kalibreringsproblem (som vi diskuterede tidligere i denne samtale — RF's overconfidence for kvinder), den løser kun et beslutnings-problem (hvor mange bliver klassificeret positivt).

2. "ExponentiatedGradient trains a randomized classifier under a constraint"

I modsætning til ThresholdOptimizer (som er post-processing — justerer en allerede færdigtrænet model), er ExponentiatedGradient en in-processing-metode — den træner en helt ny model fra bunden, med en indbygget fairness-begrænsning (her: EqualizedOdds()).

"Randomized" er den vigtige detalje: den resulterende klassifikator er ikke én fast model, men en sandsynlighedsfordeling over flere modeller — for en given person kan den (i teorien) give forskellige forudsigelser ved gentagne kørsler, fordi den reelt "trækker lod" mellem flere underliggende klassifikatorer for at opnå den ønskede fairness-balance i gennemsnit over hele populationen. Det er en anden type løsning end ThresholdOptimizer's deterministiske gruppe-specifikke thresholds.

3. "Equalized odds is evaluated relative to the label supplied during fitting"

Dette er den vigtigste advarsel til jeres situation. Se på jeres egen kode:

python
post_eo.fit(X_train, y_train, sensitive_features=A_train)

Her er y_train = train["income_observed"] — I fitter ThresholdOptimizer til at opnå equalized odds specifikt målt mod income_observed. Det betyder: metoden "ser" kun uligheder i forhold til den label, den fik at vide om under .fit() — den ved intet om income_reference overhovedet.

Konsekvens, som I faktisk allerede har observeret i jeres egne resultater: ThresholdOptimizer gjorde FPR ratio fantastisk mod obs (0.937), men kun middelmådig mod ref (0.805) — fordi den blev optimeret til at "rette" den forkerte, korrupte version af uligheden. Den lærte at kompensere for en skævhed, der delvist var et artefakt af label-korruptionen, ikke den reelle underliggende skævhed.

Hvorfor kurset fortæller jer dette eksplicit: Det er en generel, vigtig pointe i fairness-praksis: enhver postprocessing/in-processing fairness-metode er kun så god som den label, I fodrer den med under træning/fitting. Hvis jeres eneste tilgængelige label er korrupt, vil selv den "korrekte" brug af avancerede fairness-værktøjer som ThresholdOptimizer eller ExponentiatedGradient ikke garantere reel fairness — de garanterer kun fairness i forhold til den label, de blev vist.

In [177]:
from fairlearn.postprocessing import ThresholdOptimizer

post_eo = ThresholdOptimizer(
    estimator=model,               # the already-fitted Week 2 pipeline, unchanged
    constraints="equalized_odds",  # per the hint: this is what we covered in week 2
    predict_method="predict_proba",
    prefit=True,                   # tells it not to refit the base estimator
)
post_eo.fit(X_train, y_train, sensitive_features=A_train)

pred_eo = post_eo.predict(X_val, sensitive_features=A_val, random_state=RANDOM_STATE)
summarize_candidate("2. ThresholdOptimizer (equalized odds)", pred_eo, results)

In [178]:
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import EqualizedOdds, ExponentiatedGradient
from sklearn.linear_model import LogisticRegression

# TODO: choose which validation label is appropriate for fitting a postprocessor.
# post = ThresholdOptimizer(estimator=base_model, constraints="equalized_odds", predict_method="predict_proba")
# post.fit(X_val, y_val_..., sensitive_features=A_val)

# TODO: implement, justify, and audit Fairlearn interventions.

In [179]:
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds

exp_grad = ExponentiatedGradient(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    constraints=EqualizedOdds(),
)

X_train_pre = pre.fit_transform(X_train)   # ExponentiatedGradient needs numeric input directly
X_val_pre = pre.transform(X_val)

exp_grad.fit(X_train_pre, y_train, sensitive_features=A_train)
pred_expgrad = exp_grad.predict(X_val_pre)

summarize_candidate("3. ExponentiatedGradient (equalized odds, in-processing)", pred_expgrad, results)

In [180]:
batch_weight_map = {"B01": 1.5, "B02": 1.5, "B03": 1.5,   # cleanest batches — trust more
                     "B04": 1.0, "B05": 1.0,
                     "B06": 0.5, "B07": 0.5, "B08": 0.5}   # most corrupted — trust less
sample_weight = B_train.map(batch_weight_map).values

reweighted = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
reweighted.fit(X_train, y_train, clf__sample_weight=sample_weight)

pred_reweighted = (reweighted.predict_proba(X_val)[:, 1] >= 0.5).astype(int)
summarize_candidate("4. Batch-reweighted retrain", pred_reweighted, results)

n_reweighted = int((sample_weight != 1.0).sum())
print(f"Cost: {n_reweighted}/{len(sample_weight)} training rows ({n_reweighted/len(sample_weight):.1%}) reweighted.")


Cost: 13915/18000 training rows (77.3%) reweighted.


In [181]:
# Filtrer træningsdata til kun tidlige, mere pålidelige batches - removing only most under labeled batch
clean_batches_b07 = ["B01", "B02", "B03", "B04", "B05", "B06", "B08"]
train_no_b07= train[train["SURVEY_BATCH"].isin(clean_batches_b07)]

X_train_no_b07 = train_no_b07[FEATURE_COLUMNS]
y_train_no_b07 = train_no_b07["income_observed"]

model_no_b07 = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
model_no_b07.fit(X_train_no_b07, y_train_no_b07)

pred_no_b07= (model_no_b07.predict_proba(X_val)[:, 1] >= 0.5).astype(int)
summarize_candidate("5. Most underlabeled batch removed (B07)", pred_no_b07, results)

n_removed = len(train) - len(train_no_b07)
print(f"Cost: {n_removed}/{len(train)} training rows ({n_removed/len(train):.1%}) removed.")

Cost: 2681/18000 training rows (14.9%) removed.


In [182]:
# Filtrer træningsdata til kun tidlige, mere pålidelige batches - removing only most under labeled batch
clean_batches_b0708 = ["B01", "B02", "B03", "B04", "B05", "B06"]
train_no_b0708= train[train["SURVEY_BATCH"].isin(clean_batches_b0708)]

X_train_no_b0708 = train_no_b0708[FEATURE_COLUMNS]
y_train_no_b0708 = train_no_b0708["income_observed"]

model_no_b0708 = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
model_no_b0708.fit(X_train_no_b0708, y_train_no_b0708)

pred_no_b0708= (model_no_b0708.predict_proba(X_val)[:, 1] >= 0.5).astype(int)
summarize_candidate("5. Two most underlabeled batch removed (B07 and B08)", pred_no_b0708, results)

n_removed = len(train) - len(train_no_b0708)
print(f"Cost: {n_removed}/{len(train)} training rows ({n_removed/len(train):.1%}) removed.")

Cost: 5372/18000 training rows (29.8%) removed.


In [183]:
# Filtrer træningsdata til kun tidlige, mere pålidelige batches - removing only most under labeled batch
clean_batches = ["B01", "B02", "B03"]
train_filtered = train[train["SURVEY_BATCH"].isin(clean_batches)]

X_train_filtered = train_filtered[FEATURE_COLUMNS]
y_train_filtered = train_filtered["income_observed"]

model_filtered = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
model_filtered.fit(X_train_filtered, y_train_filtered)

pred_filtered = (model_filtered.predict_proba(X_val)[:, 1] >= 0.5).astype(int)
summarize_candidate("5. Only early batches", pred_filtered, results)

n_removed = len(train) - len(train_filtered)
print(f"Cost: {n_removed}/{len(train)} training rows ({n_removed/len(train):.1%}) removed.")

Cost: 12157/18000 training rows (67.5%) removed.


In [184]:
import pandas as pd
pd.set_option("display.width", 160)

results_df = pd.DataFrame(results).set_index("candidate")
display(results_df.round(3))

,accuracy_obs,accuracy_ref,selection_rate_ratio_obs,TPR_ratio_obs,FPR_ratio_obs,PPV_ratio_obs,NPV_ratio_obs,selection_rate_ratio_ref,TPR_ratio_ref,FPR_ratio_ref,PPV_ratio_ref,NPV_ratio_ref
candidate,,,,,,,,,,,,
1. Baseline (unmitigated),0.762,0.770,0.764,0.901,0.678,0.604,0.849,0.764,0.947,0.886,0.800,0.877
2. ThresholdOptimizer (equalized odds),0.743,0.752,0.755,0.971,0.937,0.658,0.826,0.755,0.980,0.805,0.873,0.848
"3. ExponentiatedGradient (equalized odds, in-processing)",0.734,0.742,0.792,0.928,0.777,0.600,0.825,0.792,0.980,0.974,0.799,0.855
4. Batch-reweighted retrain,0.754,0.773,0.840,0.947,0.708,0.577,0.880,0.840,0.990,0.855,0.761,0.898
5. Most underlabeled batch removed (B07),0.760,0.773,0.798,0.928,0.697,0.595,0.860,0.798,0.978,0.892,0.791,0.881
5. Two most underlabeled batch removed (B07 and B08),0.756,0.774,0.815,0.927,0.728,0.582,0.881,0.815,0.977,0.897,0.774,0.898
5. Only early batches,0.702,0.733,0.945,0.998,0.747,0.543,0.926,0.945,0.977,0.817,0.699,0.933


In [185]:
from sklearn.metrics import brier_score_loss

calibration_results = []

# For metoder med brugbare predict_proba scores
score_map = {
    "1. Baseline": model.predict_proba(X_val)[:, 1],
    "4. Batch-reweighted": reweighted.predict_proba(X_val)[:, 1],
    "5. Early-batch-only": model_filtered.predict_proba(X_val)[:, 1],
}

for name, s in score_map.items():
    brier_obs = brier_score_loss(val_merged["income_observed"], s)
    brier_ref = brier_score_loss(val_merged["income_reference"], s)
    calibration_results.append({"candidate": name, "Brier_obs": brier_obs, "Brier_ref": brier_ref})

calib_df = pd.DataFrame(calibration_results).set_index("candidate").round(4)
display(calib_df)

,Brier_obs,Brier_ref
candidate,,
1. Baseline,0.1648,0.1663
4. Batch-reweighted,0.1602,0.1566
5. Early-batch-only,0.1973,0.1808


In [186]:
# Byg den fulde sammenlignings-tabel: performance + fairness + calibration + cost
final_comparison = pd.DataFrame({
    "Accuracy":       [0.770, 0.752, 0.735, 0.773, 0.733],
    "Selection ratio":[0.764, 0.755, 0.805, 0.840, 0.945],
    "TPR ratio":      [0.947, 0.980, 0.988, 0.990, 0.977],
    "FPR ratio":      [0.886, 0.805, 0.966, 0.855, 0.817],
}, index=["1. Baseline", "2. ThresholdOptimizer", "3. ExponentiatedGradient", 
          "4. Batch-reweighted", "5. Early-batch-only"])

# Tilføj Brier score (kør denne, hvis I ikke allerede har tallene)
brier_baseline = brier_score_loss(val_merged["income_reference"], model.predict_proba(X_val)[:, 1])
brier_reweighted = brier_score_loss(val_merged["income_reference"], reweighted.predict_proba(X_val)[:, 1])
brier_filtered = brier_score_loss(val_merged["income_reference"], model_filtered.predict_proba(X_val)[:, 1])

final_comparison["Brier (ref)"] = [brier_baseline, brier_baseline, np.nan, brier_reweighted, brier_filtered]

# Tilføj Cost som tekst-kolonne
final_comparison["Cost"] = ["—", "−1.8pp acc", "−3.5pp acc", 
                              f"{n_reweighted}/{len(sample_weight)} rows reweighted", 
                              "68% rows removed"]

display(final_comparison.round(3))

,Accuracy,Selection ratio,TPR ratio,FPR ratio,Brier (ref),Cost
1. Baseline,0.770,0.764,0.947,0.886,0.166,—
2. ThresholdOptimizer,0.752,0.755,0.980,0.805,0.166,−1.8pp acc
3. ExponentiatedGradient,0.735,0.805,0.988,0.966,NaN,−3.5pp acc
4. Batch-reweighted,0.773,0.840,0.990,0.855,0.157,13915/18000 rows reweighted
5. Early-batch-only,0.733,0.945,0.977,0.817,0.181,68% rows removed


| Candidate | Acc | Sel. | TPR | FPR |
|---|---|---|---|---|
| 1. Baseline | 0.770 | 0.764 | 0.947 | 0.886 |
| 2. ThresholdOpt | 0.752 | 0.755 | 0.980 | 0.805 |
| 3. ExpGradient | 0.735 | 0.805 | 0.988 | 0.966 |
| 4. Reweighted | 0.773 | 0.840 | 0.990 | 0.855 |
| 5. Early-batch | 0.733 | 0.945 | 0.977 | 0.817 |

Compare at least these score pipelines:

1. uncalibrated model;
2. calibration fitted to observed validation labels;
3. calibration fitted to reference validation labels.

Evaluate each against both observed and reference test labels. Keep calibration separate from decision postprocessing. Explain why a score can be calibrated against one label and miscalibrated against another.

In [187]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Calibration is fit on the VALIDATION split -- disjoint from training -- using two
# different label sources. Pipeline C is "calibration only": no classifier is trained
# on income_reference, only a monotonic recalibration of the already-fitted scores.
calib_obs = CalibratedClassifierCV(estimator=FrozenEstimator(model), method="sigmoid")
calib_obs.fit(val_merged[FEATURE_COLUMNS], val_merged["income_observed"])

calib_ref = CalibratedClassifierCV(estimator=FrozenEstimator(model), method="sigmoid")
calib_ref.fit(val_merged[FEATURE_COLUMNS], val_merged["income_reference"])

print("Both calibration pipelines fit.")

Both calibration pipelines fit.


In [188]:
def expected_calibration_error(y_true, y_score, n_bins=10, min_bin_count=5):
    """Bins with fewer than min_bin_count points are skipped -- a reliability point
    from a handful of examples isn't trustworthy enough to include."""
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(y_score, bins[1:-1])
    ece, total = 0.0, len(y_true)
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() < min_bin_count:
            continue
        ece += (mask.sum() / total) * abs(y_true[mask].mean() - y_score[mask].mean())
    return ece

X_test = test_obs[FEATURE_COLUMNS]
y_test_obs = test_obs["income_observed"]

pipelines = {
    "A. Uncalibrated": model.predict_proba(X_test)[:, 1],
    "B. Calibrated on val income_observed": calib_obs.predict_proba(X_test)[:, 1],
    "C. Calibrated on val income_reference": calib_ref.predict_proba(X_test)[:, 1],
}

rows = []
for name, s in pipelines.items():
    rows.append({
        "pipeline": name,
        "AUROC": roc_auc_score(y_test_obs, s),
        "log_loss": log_loss(y_test_obs, s),
        "brier": brier_score_loss(y_test_obs, s),
        "ECE": expected_calibration_error(y_test_obs.values, s),
    })
display(pd.DataFrame(rows).set_index("pipeline").round(4))

,AUROC,log_loss,brier,ECE
pipeline,,,,
A. Uncalibrated,0.8162,0.5004,0.1657,0.0605
B. Calibrated on val income_observed,0.8162,0.4889,0.1611,0.0184
C. Calibrated on val income_reference,0.8162,0.4935,0.1624,0.0346


In [189]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

# Current sklearn pattern for a pre-fitted estimator. The calibration set
# must be disjoint from the model-fitting set:
# calibrated_obs = CalibratedClassifierCV(
#     estimator=FrozenEstimator(base_model), method="sigmoid"
# )
# calibrated_obs.fit(X_val, y_val_observed)

# TODO: implement all three pipelines, reliability plots, ECE, and log loss.

In [190]:
# Byg den fulde sammenlignings-tabel: performance + fairness + calibration + cost
final_comparison = pd.DataFrame({
    "Accuracy":       [0.770, 0.752, 0.735, 0.773, 0.733, 0.773],
    "Selection ratio":[0.764, 0.755, 0.805, 0.840, 0.945, 0.798],
    "TPR ratio":      [0.947, 0.980, 0.988, 0.990, 0.977, 0.978],
    "FPR ratio":      [0.886, 0.805, 0.966, 0.855, 0.817, 0.892],
}, index=["1. Baseline", "2. ThresholdOptimizer", "3. ExponentiatedGradient", 
          "4. Batch-reweighted", "5. Early-batch-only", "6. Remove B07 only"])

# Tilføj Brier score
brier_baseline = brier_score_loss(val_merged["income_reference"], model.predict_proba(X_val)[:, 1])
brier_reweighted = brier_score_loss(val_merged["income_reference"], reweighted.predict_proba(X_val)[:, 1])
brier_filtered = brier_score_loss(val_merged["income_reference"], model_filtered.predict_proba(X_val)[:, 1])
brier_no_b07 = brier_score_loss(val_merged["income_reference"], model_no_b07.predict_proba(X_val)[:, 1])

final_comparison["Brier (ref)"] = [brier_baseline, brier_baseline, np.nan, brier_reweighted, brier_filtered, brier_no_b07]

# Tilføj Cost som tekst-kolonne
final_comparison["Cost"] = ["—", "−1.8pp acc", "−3.5pp acc", 
                              f"{n_reweighted}/{len(sample_weight)} rows reweighted", 
                              "68% rows removed",
                              "2681/18000 rows removed (14.9%)"]

display(final_comparison.round(3))

,Accuracy,Selection ratio,TPR ratio,FPR ratio,Brier (ref),Cost
1. Baseline,0.770,0.764,0.947,0.886,0.166,—
2. ThresholdOptimizer,0.752,0.755,0.980,0.805,0.166,−1.8pp acc
3. ExponentiatedGradient,0.735,0.805,0.988,0.966,NaN,−3.5pp acc
4. Batch-reweighted,0.773,0.840,0.990,0.855,0.157,13915/18000 rows reweighted
5. Early-batch-only,0.733,0.945,0.977,0.817,0.181,68% rows removed
6. Remove B07 only,0.773,0.798,0.978,0.892,0.160,2681/18000 rows removed (14.9%)


Remove B07 and B08

In [197]:
# Byg den fulde sammenlignings-tabel: performance + fairness + calibration + cost
final_comparison = pd.DataFrame({
    "Accuracy":       [0.770, 0.752, 0.735, 0.773, 0.774],
    "Selection ratio":[0.764, 0.755, 0.805, 0.840, 0.815],
    "TPR ratio":      [0.947, 0.980, 0.988, 0.990, 0.977],
    "FPR ratio":      [0.886, 0.805, 0.966, 0.855, 0.897],
}, index=["1. Baseline", "2. ThresholdOptimizer", "3. ExponentiatedGradient", 
          "4. Batch-reweighted", "5. Remove B07+B08"])

# Tilføj Brier score
brier_baseline = brier_score_loss(val_merged["income_reference"], model.predict_proba(X_val)[:, 1])
brier_reweighted = brier_score_loss(val_merged["income_reference"], reweighted.predict_proba(X_val)[:, 1])
brier_no_b07_b08 = brier_score_loss(val_merged["income_reference"], model_no_b0708.predict_proba(X_val)[:, 1])

final_comparison["Brier (ref)"] = [brier_baseline, brier_baseline, np.nan, brier_reweighted, brier_no_b07_b08]

# Tilføj Cost som tekst-kolonne
n_removed_b07_b08 = len(train) - len(train_no_b0708)
final_comparison["Cost"] = ["—", "−1.8pp acc", "−3.5pp acc", 
                              f"{n_reweighted}/{len(sample_weight)} rows reweighted", 
                              f"{n_removed_b07_b08}/{len(train)} rows removed ({n_removed_b07_b08/len(train):.1%})"]

display(final_comparison.round(3))

,Accuracy,Selection ratio,TPR ratio,FPR ratio,Brier (ref),Cost
1. Baseline,0.770,0.764,0.947,0.886,0.166,—
2. ThresholdOptimizer,0.752,0.755,0.980,0.805,0.166,−1.8pp acc
3. ExponentiatedGradient,0.735,0.805,0.988,0.966,NaN,−3.5pp acc
4. Batch-reweighted,0.773,0.840,0.990,0.855,0.157,13915/18000 rows reweighted
5. Remove B07+B08,0.774,0.815,0.977,0.897,0.156,5372/18000 rows removed (29.8%)


## 5. Calibration comparison

In [191]:
test_ref = pd.read_csv(DATA_DIR / "week3_test_reference.csv")  # touched here, and only here

X_test_final = test_ref[FEATURE_COLUMNS]
A_test_final = test_ref["SEX_group"]

pred_final = post_eo.predict(X_test_final, sensitive_features=A_test_final, random_state=RANDOM_STATE)
score_final = model.predict_proba(X_test_final)[:, 1]  # base scores, for AUROC/log-loss context

print(f"Final predictions generated for {len(pred_final)} test rows.")

Final predictions generated for 6000 test rows.


In [192]:
ratio_final_obs = run_audit(test_ref["income_observed"], score_final, pred_final, A_test_final, "Y^observed (FINAL TEST)")
ratio_final_ref = run_audit(test_ref["income_reference"], score_final, pred_final, A_test_final, "Y^reference (FINAL TEST)")

===================== Evaluated against Y^observed (FINAL TEST) =====================

-- Utility --
Accuracy:          0.7440
Balanced accuracy: 0.6628
AUROC:             0.8162
Log loss:          0.5004

-- By group --
            count  selection_rate     TPR     FPR     PPV     NPV
SEX_group                                                        
Female     2811.0          0.1537  0.3782  0.0886  0.5532  0.8348
Male       3189.0          0.2477  0.4412  0.1039  0.7595  0.6832

-- Fairness ratios (min/max across groups) --
Independence -- selection_rate ratio: 0.6204
Separation   -- TPR ratio: 0.8572   FPR ratio: 0.8526
Sufficiency  -- PPV ratio: 0.7284   NPV ratio: 0.8184

-- Calibration --
Brier score: 0.1657

===================== Evaluated against Y^reference (FINAL TEST) =====================

-- Utility --
Accuracy:          0.7365
Balanced accuracy: 0.6687
AUROC:             0.8408
Log loss:          0.5039

-- By group --
            count  selection_rate     TPR     FPR    

In [193]:
R_test_final = test_ref["RACE_group"]
inter_final = A_test_final.astype(str) + " | " + R_test_final.astype(str)
counts_final = inter_final.value_counts()

print("Intersectional group sizes (SEX x RACE), test set:")
print(counts_final)
print(f"\nGroups with n >= 20 (usable for interpretation): {(counts_final >= 20).sum()} of {len(counts_final)}")

Intersectional group sizes (SEX x RACE), test set:
Male | White                     2518
Female | White                   2129
Female | Black                    299
Male | Black                      272
Male | Asian                      168
Male | Other race                 162
Female | Asian                    147
Female | Other race               143
Female | Two or more races         71
Male | Two or more races           53
Female | American Indian           18
Male | American Indian              9
Male | AIAN specified/other         4
Male | Pacific Islander             3
Female | AIAN specified/other       3
Female | Pacific Islander           1
Name: count, dtype: int64

Groups with n >= 20 (usable for interpretation): 10 of 16


In [194]:
rng = np.random.default_rng(RANDOM_STATE)
n = len(test_ref)
y_ref_arr = test_ref["income_reference"].values
A_arr = A_test_final.values

boot = {"selection_rate": [], "TPR": [], "FPR": [], "PPV": []}
for _ in range(500):
    idx = rng.integers(0, n, n)  # resample rows with replacement
    mf_b = MetricFrame(
        metrics={"selection_rate": selection_rate, "TPR": recall_score,
                 "FPR": false_positive_rate_metric, "PPV": precision_score},
        y_true=y_ref_arr[idx], y_pred=pred_final[idx],
        sensitive_features=pd.Series(A_arr[idx]),
    )
    r = mf_b.ratio(method="between_groups")
    for k in boot:
        boot[k].append(r[k])

print("Bootstrap 95% CIs, ratio vs. income_reference (500 resamples):")
for k, vals in boot.items():
    lo, hi = np.percentile(vals, [2.5, 97.5])
    print(f"  {k} ratio: point={ratio_final_ref[k]:.3f}   95% CI=[{lo:.3f}, {hi:.3f}]")

Bootstrap 95% CIs, ratio vs. income_reference (500 resamples):
  selection_rate ratio: point=0.620   95% CI=[0.553, 0.686]
  TPR ratio: point=0.866   95% CI=[0.767, 0.954]
  FPR ratio: point=0.653   95% CI=[0.524, 0.801]
  PPV ratio: point=0.901   95% CI=[0.836, 0.961]


## 5. Final locked test evaluation

Select the final candidates before examining reference test results. Then produce evaluate a comparison with:

- performance against $Y^{obs}$ and $Y^{ref}$;
- sex-specific and intersectional counts;
- selection-rate gap;
- TPR and FPR gaps;
- PPV and NPV gaps;
- AUROC, log loss, and calibration error;
- bootstrap intervals for key gaps;
- number of training examples removed/reweighted;
- features required at deployment.
- etc

Reweighted data

In [195]:
13915/18000


0.7730555555555556

In [196]:
X_test_final = test_ref[FEATURE_COLUMNS]
score_test_final = reweighted.predict_proba(X_test_final)[:, 1]
pred_test_final = (score_test_final >= 0.5).astype(int)
A_test_final = test_ref["SEX_group"]

ratio_test_obs = run_audit(test_ref["income_observed"], score_test_final, pred_test_final, A_test_final, "Y^observed (FINAL TEST)")
ratio_test_ref = run_audit(test_ref["income_reference"], score_test_final, pred_test_final, A_test_final, "Y^reference (FINAL TEST)")

===================== Evaluated against Y^observed (FINAL TEST) =====================

-- Utility --
Accuracy:          0.7650
Balanced accuracy: 0.7097
AUROC:             0.8164
Log loss:          0.4889

-- By group --
            count  selection_rate     TPR     FPR     PPV     NPV
SEX_group                                                        
Female     2811.0          0.2287  0.5364  0.1395  0.5272  0.8649
Male       3189.0          0.2970  0.5493  0.1093  0.7888  0.7266

-- Fairness ratios (min/max across groups) --
Independence -- selection_rate ratio: 0.7703
Separation   -- TPR ratio: 0.9766   FPR ratio: 0.7838
Sufficiency  -- PPV ratio: 0.6684   NPV ratio: 0.8401

-- Calibration --
Brier score: 0.1612

===================== Evaluated against Y^reference (FINAL TEST) =====================

-- Utility --
Accuracy:          0.7698
Balanced accuracy: 0.7226
AUROC:             0.8409
Log loss:          0.4802

-- By group --
            count  selection_rate     TPR     FPR    

## Questions to think about!

1. Which Week 2 conclusions were robust to the reference label audit?
2. Which apparent performance or fairness results were artifacts of $Y^{obs}$?
3. How did the suspected issue X affect in-distribution validation and shifted test performance?
4. How methods/framework helped you find issues in the data? How do you know?
5. Which mitigation improved reference performance, and what did it cost?
6. Did a parity intervention improve the chosen harm-related metric against the correct label?
7. How did observed-label and reference-label calibration differ?
8. Why can equalized odds and calibration conflict when group base rates differ?
9. What evidence would be required before changing real labels or deploying group-specific thresholds?
10. What remains unvalidated?

# 📌 Final Submission for Project 1: A0 Poster

You are instructed to submit an A0 poster summarizing  findings across the three weeks.

## 📋 Recommended Poster Structure

### Introduction & Setup (very breifly)

* The prediction task, dataset, and outcome.
* The critical distinction between $Y^{obs}$ (the potential corrupted label) and $Y^{ref}$ (the clean ground-truth label).

### Week 1: Baseline Fairness Audit

* Overall and group-specific predictive performance.
* Fairness metrics considered and the real-world trade-offs observed.

### Week 2: Blind Audit (Observed Labels)

* You must explain initial hypotheses (e.g., Data collection flaws? Group-dependent errors? Spurious correlations?).
* What audit methods they used, their strongest evidence, and whether they trust derived scores as ground truth.

### Week 3: Reference Label Evaluation and Mitigation

* Reveal the hidden data problem using $Y^{ref}$ and quantify the disagreement between the observed and reference labels.
* Discuss whether your Week 2 hypotheses were correct.
* Mitigation: Compare a justified set of interventions. Report the effect on performance, fairness, calibration, and any adverse effects.
* Final Selection: Present the final model selection and its evaluation on the reference test set.
* Discussion and Conclusion

### Important

* Use visual evidence (tables/figures) throughout the poster. Focus on interpretation over code.

* Report relevant negative or inconclusive findings. What remains unvalidated?

# **Poster**

## 1. Reveal the hidden data problem using $Y^{ref}$ and quantify the disagreement between the observed and reference labels.

**Disagreement rate by sex**

| Sex | Total observations | Disagreement rate | Disagreeing rows |
|---|---|---|---|
| Female | 2812 | 7.25% | 204 |
| Male | 3188 | 1.19% | 38 |

**Disagreement rate by survey batch**

| Batch | Total observations | Disagreement rate | Disagreeing rows |
|---|---|---|---|
| B01 | 641 | 2.50% | 16 |
| B02 | 620 | 3.23% | 20 |
| B03 | 642 | 2.65% | 17 |
| B04 | 681 | 4.99% | 34 |
| B05 | 727 | 4.26% | 31 |
| B06 | 963 | 4.15% | 40 |
| B07 | 847 | 5.31% | 45 |
| B08 | 879 | 4.44% | 39 |

**Under- vs. over-labeling, by sex**

| Sex | Total observations | Under-labeled rate | Under-labeled rows | Over-labeled rate | Over-labeled rows |
|---|---|---|---|---|---|
| Female | 2812 | 6.65% | 187 | 0.60% | 17 |
| Male | 3188 | 0.78% | 25 | 0.41% | 13 |

**Under-labeling rate by batch and sex**

| Batch | Female n | Female under-labeled rate | Male n | Male under-labeled rate |
|---|---|---|---|---|
| B01 | 272 | 2.94% | 369 | 0.81% |
| B02 | 250 | 5.20% | 370 | 0.54% |
| B03 | 248 | 4.44% | 394 | 0.25% |
| B04 | 317 | 5.99% | 364 | 2.47% |
| B05 | 378 | 6.88% | 349 | 0.57% |
| B06 | 493 | 7.71% | 470 | 0.21% |
| B07 | 442 | 9.05% | 405 | 0.74% |
| B08 | 412 | 7.77% | 467 | 0.86% |

## 2. Discuss whether your Week 2 hypotheses were correct.

## 3. Mitigation: Compare a justified set of interventions. Report the effect on performance, fairness, calibration, and any adverse effects.